# Self-attention step by step
1. The first-principles version: weights from dot products and softmax, then a weighted sum.
   Real 50-number word embeddings learned from 116,830 English sentences (as in the previous Note).
2. Word by word, then all words at once with matrices: the same numbers.
3. Query, key and value vectors from three matrices W_Q, W_K, W_V.
4. On IMDB reviews: the same frozen embeddings with fixed self-attention (no parameters) vs
   self-attention with learned W_Q, W_K, W_V. 3 seeds each.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import collections
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds
import tensorflow as tf
import keras

np.set_printoptions(precision=3, suppress=True)


def ppmi_svd(pairs_rows, pairs_cols, n, k=50):
    """Static embeddings: PPMI-weighted co-occurrence counts, compressed to k numbers by the SVD, unit length."""
    C = sp.coo_matrix((np.ones(len(pairs_rows), np.float32), (pairs_rows, pairs_cols)), shape=(n, n)).tocsr()
    total = C.sum()
    p_word = np.asarray(C.sum(1)).ravel() / total
    p_ctx = (np.asarray(C.sum(0)).ravel() / total) ** 0.75
    p_ctx /= p_ctx.sum()
    C = C.tocoo()
    pmi = np.log((C.data / total) / (p_word[C.row] * p_ctx[C.col]))
    keep = pmi > 0
    U, S, _ = svds(sp.csr_matrix((pmi[keep], (C.row[keep], C.col[keep])), shape=(n, n)), k=k, random_state=0)
    E = U * np.sqrt(S)
    return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)


def softmax_rows(S):
    W = np.exp(S - S.max(1, keepdims=True))
    return W / W.sum(1, keepdims=True)

I0000 00:00:1791074481.396018 3168121 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Static embeddings of 116,830 English sentences

In [2]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = sorted(set(l.split("\t")[0] for l in open(f"{path}/fra.txt", encoding="utf-8")))
sents = [re.findall(r"[a-z']+", l.lower()) for l in lines]
count = collections.Counter(w for s in sents for w in s)
vocab = [w for w, c in count.items() if c >= 5]
idx = {w: i for i, w in enumerate(vocab)}
rows, cols = [], []
for s in sents:
    ids = [idx[w] for w in s if w in idx]
    for a in range(len(ids)):
        for b in range(max(0, a - 4), min(len(ids), a + 5)):
            if a != b:
                rows.append(ids[a]); cols.append(ids[b])
E = ppmi_svd(rows, cols, len(vocab))
print(E.shape)

(5287, 50)


## Dot product as similarity: the three 2-D vectors of the text

In [3]:
a, b, c = np.array([6, 1]), np.array([4, 2]), np.array([1, 5])
print("a.b =", a @ b, "  a.c =", a @ c)

a.b = 26   a.c = 11


## One word: the new vector of "bank" in "money bank grows"

In [4]:
sentence = "money bank grows".split()
X = np.array([E[idx[w]] for w in sentence])
s = X[1] @ X.T                                  # dot products of bank with money, bank, grows
w = np.exp(s) / np.exp(s).sum()                 # softmax
y_bank = w @ X                                  # weighted sum of the three embeddings
print("scores s21, s22, s23:", s.round(3))
print("weights w21, w22, w23:", w.round(3), " sum", w.sum().round(3))
print("first 5 numbers of y_bank:", y_bank[:5])

scores s21, s22, s23: [ 0.399  1.    -0.021]
weights w21, w22, w23: [0.287 0.524 0.189]  sum 1.0
first 5 numbers of y_bank: [-0.034 -0.059  0.025  0.046 -0.033]


## Every word, one at a time (a loop) and all at once (matrices)

In [5]:
def self_attention_loop(X):
    Y = np.zeros_like(X)
    for i in range(len(X)):
        scores = np.array([X[i] @ X[j] for j in range(len(X))])
        weights = np.exp(scores) / np.exp(scores).sum()
        Y[i] = sum(weights[j] * X[j] for j in range(len(X)))
    return Y


def self_attention_matrix(X):
    return softmax_rows(X @ X.T) @ X


weights_simple = {}
for text in ("money bank grows", "river bank flows"):
    X = np.array([E[idx[w]] for w in text.split()])
    assert np.allclose(self_attention_loop(X), self_attention_matrix(X))
    weights_simple[text] = softmax_rows(X @ X.T)
    print(text, "\n", weights_simple[text])
print("loop == matrix form: True")
pd.concat({k: pd.DataFrame(v, index=k.split(), columns=k.split()) for k, v in weights_simple.items()}).round(3).to_csv(
    "data/weights_simple.csv")

money bank grows 
 [[0.529 0.29  0.181]
 [0.287 0.524 0.189]
 [0.201 0.212 0.587]]
river bank flows 
 [[0.45  0.209 0.341]
 [0.251 0.539 0.21 ]
 [0.353 0.182 0.465]]
loop == matrix form: True


The matrix form on a longer sentence: shapes.

In [6]:
X = np.array([E[idx[w]] for w in "i put my money in the bank".split()])
print("X", X.shape, " X X^T", (X @ X.T).shape, " output", self_attention_matrix(X).shape)

X (7, 50)  X X^T (7, 7)  output (7, 50)


## Query, key and value: a tiny worked example in 2-D
Row vectors times matrices, q = e W_Q (the same convention as Keras).

In [7]:
e = np.array([1.0, 2.0])
W_Q = np.array([[1.0, 0.0], [1.0, 1.0]])
W_K = np.array([[0.0, 1.0], [1.0, 0.0]])
W_V = np.array([[2.0, 0.0], [0.0, 0.5]])
print("q =", e @ W_Q, " k =", e @ W_K, " v =", e @ W_V)

q = [3. 2.]  k = [2. 1.]  v = [2. 1.]


The full version with matrices: Q = X W_Q, K = X W_K, V = X W_V, then softmax(Q K^T) V.
The same W matrices serve every word. Random matrices here; training sets them.

In [8]:
rng = np.random.default_rng(0)
X = np.array([E[idx[w]] for w in "money bank grows".split()])
Wq, Wk, Wv = (rng.normal(0, 0.3, (50, 50)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
Y = softmax_rows(Q @ K.T) @ V
Y_loop = np.array([sum(w_ij * V[j] for j, w_ij in enumerate(softmax_rows((Q[i] @ K.T)[None])[0])) for i in range(3)])
assert np.allclose(Y, Y_loop)
print("Q, K, V:", Q.shape, K.shape, V.shape, " Y:", Y.shape, " loop == matrix: True")

Q, K, V: (3, 50) (3, 50) (3, 50)  Y: (3, 50)  loop == matrix: True


## Fixed vs learned self-attention on IMDB
Static embeddings learned from the 25,000 training reviews themselves (same PPMI + SVD recipe, words
within 2 positions), then frozen. Model: embeddings -> self-attention -> average over the words of the
review -> one sigmoid node. Fixed: no parameters inside self-attention. Learned: W_Q, W_K, W_V
(50 x 50 each, started at the identity, so both models start from exactly the same function).

In [9]:
V_SIZE, L, d = 10000, 200, 50
(xtr, ytr), (xte, yte) = keras.datasets.imdb.load_data(num_words=V_SIZE)
xtr = keras.utils.pad_sequences(xtr, maxlen=L, padding="post", truncating="post")
xte = keras.utils.pad_sequences(xte, maxlen=L, padding="post", truncating="post")
pr, pc = [], []
for k in (1, 2):
    left, right = xtr[:, :-k].ravel(), xtr[:, k:].ravel()
    ok = (left > 3) & (right > 3)                   # skip padding, start and unknown-word markers
    pr += [left[ok], right[ok]]; pc += [right[ok], left[ok]]
E_imdb = ppmi_svd(np.concatenate(pr), np.concatenate(pc), V_SIZE).astype("float32")
E_imdb[:4] = 0
inv = {v + 3: k for k, v in keras.datasets.imdb.get_word_index().items()}
widx = {w: i for i, w in inv.items()}
for word in ("great", "terrible"):
    print(word, "->", [inv.get(i) for i in np.argsort(-(E_imdb @ E_imdb[widx[word]]))[1:7]])

great -> ['wonderful', 'fine', 'fantastic', 'excellent', 'marvelous', 'superb']
terrible -> ['horrible', 'awful', 'horrid', 'atrocious', 'horrendous', 'dreadful']


In [10]:
class SelfAttention(keras.layers.Layer):
    """softmax(Q K^T) V over the real words of each review, then the average over those words."""

    def __init__(self, learned, **kw):
        super().__init__(**kw)
        self.learned = learned

    def build(self, shape):
        if self.learned:
            self.Wq, self.Wk, self.Wv = (self.add_weight(shape=(d, d), initializer="identity", name=n) for n in "qkv")

    def weights_of(self, x):
        q, k = (x @ self.Wq, x @ self.Wk) if self.learned else (x, x)
        return tf.matmul(q, k, transpose_b=True)

    def call(self, x, ids):
        real = tf.cast(ids > 0, x.dtype)                         # 0 = padding
        scores = self.weights_of(x) + (1.0 - real[:, None, :]) * -1e9
        v = x @ self.Wv if self.learned else x
        y = tf.matmul(tf.nn.softmax(scores, -1), v)
        return tf.reduce_sum(y * real[..., None], 1) / tf.reduce_sum(real, 1, keepdims=True)


def build(learned, seed):
    keras.utils.set_random_seed(seed)
    ids = keras.Input((L,), dtype="int32")
    x = keras.layers.Embedding(V_SIZE, d, weights=[E_imdb], trainable=False)(ids)
    out = keras.layers.Dense(1, activation="sigmoid")(SelfAttention(learned)(x, ids))
    m = keras.Model(ids, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m


print("trainable parameters, fixed:", sum(np.prod(w.shape) for w in build(False, 0).trainable_weights),
      " learned:", sum(np.prod(w.shape) for w in build(True, 0).trainable_weights))

trainable parameters, fixed: 51  learned: 7551


In [11]:
results, models = [], {}
for learned in (False, True):
    for seed in range(3):
        m = build(learned, seed)
        h = m.fit(xtr, ytr, epochs=5, batch_size=64, validation_data=(xte, yte), verbose=0).history
        results.append(dict(model="learned W_Q, W_K, W_V" if learned else "fixed (no parameters)", seed=seed,
                            test_accuracy=h["val_accuracy"][-1]))
        models[(learned, seed)] = m
res = pd.DataFrame(results)
res.to_csv("data/imdb_accuracy.csv", index=False)
print(res.round(3))
res.groupby("model").test_accuracy.mean().round(3)

                   model  seed  test_accuracy
0  fixed (no parameters)     0          0.692
1  fixed (no parameters)     1          0.691
2  fixed (no parameters)     2          0.689
3  learned W_Q, W_K, W_V     0          0.834
4  learned W_Q, W_K, W_V     1          0.838
5  learned W_Q, W_K, W_V     2          0.836


model
fixed (no parameters)    0.691
learned W_Q, W_K, W_V    0.836
Name: test_accuracy, dtype: float64

## The attention weights on a real review
The first 10-word stretch of a test review that contains one of a few strong sentiment words.

In [12]:
ids_test = keras.datasets.imdb.load_data(num_words=V_SIZE)[1][0]
found = None
for r in ids_test:
    words = [inv.get(i, "?") for i in r[1:]]
    for s0 in range(len(words) - 10):
        win = words[s0:s0 + 10]
        if "?" not in win and any(wd in ("great", "terrible", "awful", "excellent", "boring", "wonderful") for wd in win) \
                and all(widx[wd] < V_SIZE for wd in win):
            found = win
            break
    if found:
        break
print(" ".join(found))
Xr = tf.constant(E_imdb[[widx[wd] for wd in found]][None])
layer = [l for l in models[(True, 0)].layers if isinstance(l, SelfAttention)][0]
W_fixed = softmax_rows(Xr[0].numpy() @ Xr[0].numpy().T)
W_learned = softmax_rows(layer.weights_of(Xr)[0].numpy())
print("fixed:\n", W_fixed, "\nlearned:\n", W_learned)
pd.concat({"fixed": pd.DataFrame(W_fixed, index=found, columns=found),
           "learned": pd.DataFrame(W_learned, index=found, columns=found)}).round(3).to_csv("data/weights_review.csv")

and the rest of the cast rendered terrible performances the
fixed:
 [[0.203 0.078 0.08  0.079 0.078 0.097 0.092 0.102 0.114 0.078]
 [0.062 0.163 0.076 0.098 0.163 0.063 0.068 0.085 0.059 0.163]
 [0.077 0.091 0.196 0.095 0.091 0.099 0.078 0.087 0.094 0.091]
 [0.075 0.116 0.094 0.194 0.116 0.068 0.07  0.083 0.068 0.116]
 [0.062 0.163 0.076 0.098 0.163 0.063 0.068 0.085 0.059 0.163]
 [0.091 0.074 0.096 0.067 0.074 0.191 0.091 0.088 0.155 0.074]
 [0.094 0.087 0.084 0.076 0.087 0.1   0.21  0.09  0.085 0.087]
 [0.095 0.098 0.084 0.081 0.098 0.087 0.08  0.188 0.09  0.098]
 [0.108 0.07  0.092 0.068 0.07  0.158 0.078 0.093 0.193 0.07 ]
 [0.062 0.163 0.076 0.098 0.163 0.063 0.068 0.085 0.059 0.163]] 
learned:
 [[0.042 0.024 0.095 0.012 0.024 0.123 0.14  0.458 0.06  0.024]
 [0.021 0.041 0.096 0.029 0.041 0.103 0.122 0.458 0.05  0.041]
 [0.011 0.005 0.181 0.01  0.005 0.097 0.077 0.575 0.033 0.005]
 [0.018 0.012 0.106 0.036 0.012 0.084 0.099 0.583 0.037 0.012]
 [0.021 0.041 0.096 0.029 0.041 0.103 